In [1]:
import os, sys
from pathlib import Path
import cortex
import nibabel as nb
import numpy as np
import matplotlib.colors as colors
import matplotlib.pyplot as pl
import time
import platform
import pickle
import pandas as pd
from matplotlib.colors import Normalize
from prf_expect.utils import io

In [2]:
surprise_analysis_type = 'TRMI-type1'

In [3]:
# Define paths and data exp parameters
settings = io.load_settings()
data_dir = Path(settings["general"]["data_dir"], "data")
tasks = settings["design"]["tasks"]
space = settings["mri"]["space"]

Loading settings from /Users/dionysus/Library/CloudStorage/OneDrive-Personal/Workbench/pRF_expect_pub/pRF_expect_analysis/prf_expect/settings.yml


In [4]:
subjects = ["sub-001", "sub-002", "sub-004", "sub-005", "sub-007", "sub-009", "sub-012"]

beta_omits = []
beta_spars = []
rsqs = []
for subject in subjects:
    analysis_result_dir = f"/Users/dionysus/Downloads/data/derivatives/prf_data/{subject}/ses-1/{surprise_analysis_type}"
    analysis_result_dir = Path(analysis_result_dir)
    tsv_name = Path.joinpath(
        data_dir,
        "derivatives",
        "prf_data",
        subject,
        "ses-1",
        "prf_fits",
        "prf_params",
        f"{subject}_ses-1_final-fit_space-{space}_model-norm_stage-iter_desc-prf_params.tsv",
    )
    params = pd.read_csv(tsv_name, sep="\t")
    rsq_sub = params["r2"].values
    rsq_sub[~np.isfinite(rsq_sub)] = 0.0
    rsq_sub = np.clip(rsq_sub, 0, None)  # enforce non-negative weights
    rsq_sub[rsq_sub==1.0] = 0
    rsqs.append(rsq_sub)
    if surprise_analysis_type == 'TRMI-type1':
    
        beta_omit_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-viol_beta_omitdm.npy"
        beta_omit_sub = np.load(beta_omit_fn, allow_pickle=True)
        beta_omits.append(beta_omit_sub)

        beta_spar_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-viol_beta_sparsedm.npy"
        beta_spar_sub = np.load(beta_spar_fn, allow_pickle=True)
        beta_spars.append(beta_spar_sub)
    if surprise_analysis_type == 'TRMI-type2':
        beta_omit_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-omit_beta_omitdm.npy"
        beta_omit_sub = np.load(beta_omit_fn, allow_pickle=True)
        beta_omits.append(beta_omit_sub)

        beta_spar_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-viol_beta_sparsedm.npy"
        beta_spar_sub = np.load(beta_spar_fn, allow_pickle=True)
        beta_spars.append(beta_spar_sub)
    if surprise_analysis_type == 'TRMI-type3':
        beta_omit_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-omit_beta_omitdm.npy"
        beta_omit_sub = np.load(beta_omit_fn, allow_pickle=True)
        beta_omits.append(beta_omit_sub)

        beta_spar_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-spar_beta_spardm.npy"
        beta_spar_sub = np.load(beta_spar_fn, allow_pickle=True)
        beta_spars.append(beta_spar_sub)
    if surprise_analysis_type == 'TRMI-type4':
        beta_omit_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-viol_beta_omitdm.npy"
        beta_omit_sub = np.load(beta_omit_fn, allow_pickle=True)
        beta_omits.append(beta_omit_sub)

        beta_spar_fn = analysis_result_dir / f"{subject}_ses-1_space-fsaverage_surprise-spar_beta_spardm.npy"
        beta_spar_sub = np.load(beta_spar_fn, allow_pickle=True)
        beta_spars.append(beta_spar_sub)

beta_omits = np.array(beta_omits, dtype=float)
beta_omits[~np.isfinite(beta_omits)] = np.nan

beta_spars = np.array(beta_spars, dtype=float)
beta_spars[~np.isfinite(beta_spars)] = np.nan

rsqs = np.array(rsqs, dtype=float)

# weighted mean with fallback to unweighted nanmean where denominator is 0
den = np.sum(rsqs, axis=0)

omit_num = np.nansum(beta_omits * rsqs, axis=0)
omit_fallback = np.nanmean(beta_omits, axis=0)
beta_omits = np.divide(
    omit_num, den, out=omit_fallback.copy(), where=den > 0
)

spar_num = np.nansum(beta_spars * rsqs, axis=0)
spar_fallback = np.nanmean(beta_spars, axis=0)
beta_spars = np.divide(
    spar_num, den, out=spar_fallback.copy(), where=den > 0
)


/var/folders/1x/nhszthp519l9hrq_yddm9yhr0000gn/T/ipykernel_22909/2463381603.py:71: RuntimeWarning: Mean of empty slice
  omit_fallback = np.nanmean(beta_omits, axis=0)
/var/folders/1x/nhszthp519l9hrq_yddm9yhr0000gn/T/ipykernel_22909/2463381603.py:77: RuntimeWarning: Mean of empty slice
  spar_fallback = np.nanmean(beta_spars, axis=0)


In [5]:
cwd = os.getcwd()
figure_result_dir = Path(cwd).parent / "figures"
print('Running on {}'.format(platform.node()))
print('Current deriv folder is {}'.format(analysis_result_dir))

print('cortex.database.default_filestore: {}'.format(cortex.database.default_filestore))
print('cortex.options.usercfg: {}'.format(cortex.options.usercfg))

Running on Smoldering-Corpse-Bar.local
Current deriv folder is /Users/dionysus/Downloads/data/derivatives/prf_data/sub-012/ses-1/TRMI-type1
cortex.database.default_filestore: /Users/dionysus/Documents/pycortex/db
cortex.options.usercfg: /Users/dionysus/Library/Application Support/pycortex/options.cfg


In [6]:

subject = "fsaverage-vis"

### Run vis_TRMI_on_visual_hierarchy.ipynb first, it will generate the TRMI_fn needed below.

In [7]:
static_imgs = False
web_view = True

In [8]:
def con_weighted_dispcx(subject, data, param_rsq, cmap='seismic', vmin=-0.3, vmax=0.3, vmin2=0, vmax2=1, rsq_thre=0.25):
    curv = cortex.db.get_surfinfo(subject)
    # Adjust curvature contrast / color. Alternately, you could work
    # with curv.data, maybe threshold it, and apply a color map. 
    curv.vmin = -1
    curv.vmax = 1
    curv.cmap = 'gray'
    curv.data = curv.data * .75 + 0.6
    # curv.data = curv.data * .50
    norm2 = Normalize(vmin2, vmax2)
    # curv = cortex.Vertex(curv.data, subject, vmin=-1,vmax=1,cmap='gray')
    # Create some display data

    # # normalize the range of param_rsq to 0 to 1
    # if param_rsq.max()-param_rsq.min()>0:
    #     param_rsq = (param_rsq-param_rsq.min())/(param_rsq.max()-param_rsq.min()-0.2)
    # else:
    #     pass
    # alpha = np.clip(norm2(param_rsq), 0, 1)
    alpha = np.clip(norm2(param_rsq), 1, 1)
    alpha[param_rsq<rsq_thre] = 0
    vx = cortex.Vertex(data, subject, cmap=cmap, vmin=vmin, vmax=vmax, )

    # Map to RGB
    vx_rgb = np.vstack([vx.raw.red.data, vx.raw.green.data, vx.raw.blue.data])
    curv_rgb = np.vstack([curv.raw.red.data, curv.raw.green.data, curv.raw.blue.data])

    
    # alpha = param_rsq
    alpha = alpha.astype(np.float32)

    # Alpha mask
    display_data = vx_rgb * alpha + curv_rgb * (1 - alpha)
    # fake_curv_rgb = np.zeros_like(curv_rgb)
    # display_data = vx_rgb * alpha + fake_curv_rgb * (1 - alpha)
    # display_data /= 255
    return display_data

In [9]:
def Vertex2D_fix(data1, data2, subject, cmap, vmin, vmax, vmin2, vmax2, roi_borders=None):
    #this provides a nice workaround for pycortex opacity issues, at the cost of interactivity    
    # Get curvature
    curv = cortex.db.get_surfinfo(subject)
    # Adjust curvature contrast / color. Alternately, you could work
    # with curv.data, maybe threshold it, and apply a color map. 
    
    #standard
    curv.data = curv.data * .75 +0.1
    #alternative
    #curv.data = np.sign(curv.data) * .25
    #HCP adjustment
    #curv.data = curv.data * -2.5# 1.25 +0.1

    
    curv = cortex.Vertex(curv.data, subject, vmin=-1,vmax=1,cmap='gray')
    
    norm2 = Normalize(vmin2, vmax2)   
    
    vx = cortex.Vertex(data1, subject, cmap=cmap, vmin=vmin, vmax=vmax)
    
    # Map to RGB
    vx_rgb = np.vstack([vx.raw.red.data, vx.raw.green.data, vx.raw.blue.data])
    
    curv_rgb = np.vstack([curv.raw.red.data, curv.raw.green.data, curv.raw.blue.data])

    
    # Pick an arbitrary region to mask out
    # (in your case you could use np.isnan on your data in similar fashion)
    alpha = np.clip(norm2(data2), 0, 1)

    # Alpha mask
    display_data = (curv_rgb * (1-alpha)) + vx_rgb * alpha

    display_data /= 255


    #print(display_data.min())
    #print(display_data.max())
    
    if roi_borders is not None:
        display_data[:,roi_borders.astype('bool')] = 0#255-display_data[:,roi_borders.astype('bool')]#0#255
    
    # Create vertex RGB object out of R, G, B channels
    return cortex.VertexRGB(*display_data, subject)  

In [10]:
print(np.nanmin(beta_spars), np.nanmax(beta_spars))
print(np.nanmin(beta_omits), np.nanmax(beta_omits))

-1.1826080809374495e+34 1.3919118981975296e+38
-1.1826080809374495e+34 1.3919118981975296e+38


In [11]:
rsq_alpha = rsqs.mean(axis=0)
rsq_alpha

array([0.07903786, 0.23686182, 0.        , ..., 0.        , 0.        ,
       0.        ], shape=(327684,))

In [12]:
rsq_alpha = np.ones_like(beta_spars)

rsq_average = rsqs.mean(axis=0)
beta_spars_viz = beta_spars
beta_spars_viz[rsq_average<0.1] = 0
beta_omit_viz = beta_omits
beta_omit_viz[rsq_average<0.1] = 0

display_spars = con_weighted_dispcx(
    subject, 
    beta_spars_viz, 
    rsq_alpha, 
    cmap='seismic', 
    vmin=-3, 
    vmax=3,
    rsq_thre=0.
)
beta_spar_cx = cortex.VertexRGB(
    *display_spars, 
    subject,
)

display_omits = con_weighted_dispcx(
    subject, 
    beta_omit_viz, 
    rsq_alpha, 
    cmap='seismic', 
    vmin=-3, 
    vmax=3,
    rsq_thre=0.
)
beta_omit_cx = cortex.VertexRGB(
    *display_omits, 
    subject,
)

    
if web_view:
    print("creating web view")
    for d, nm, depth in zip([beta_omit_cx.raw, beta_spar_cx.raw], 
                            ["cx_viol_beta_omit", "cx_viol_beta_spar"], 
                            [0, 0]):
        ds = cortex.Dataset(**{nm: d})
        handle = cortex.webgl.show(data=ds, recache=True, labels_visible=(), overlays_visible=('rois',))
        file_pattern = "{base}_{view}_{nm}.png"
        time.sleep(20.0)
        # projection parameters
        basic = dict(
            radius=300, depth=depth, specularity=0, unfold=0.5, contrast=0
        )  # projection=['orthographic'],
        # different views available, more views can be added  and the
        # existing list can be removed
        views = dict(
            myview=dict(altitude=94, azimuth=189, pivot=0),
            # lateral=dict(altitude=90.5, azimuth=181, pivot=180),
            # medial=dict(altitude=90.5, azimuth=0, pivot=180),
            # front=dict(altitude=90.5, azimuth=0, pivot=0),
            # back=dict(altitude=90.5, azimuth=181, pivot=0),
            # top=dict(altitude=0, azimuth=180, pivot=0),
            # bottom=dict(altitude=180, azimuth=0, pivot=0),
        )
        # utility functions to set the different views
        prefix = dict(
            altitude="camera.",
            azimuth="camera.",
            pivot="surface.{subject}.",
            radius="camera.",
            unfold="surface.{subject}.",
            depth="surface.{subject}.",
            specularity="surface.{subject}.",
            contrast="surface.curvature.",
        )
        _tolists = lambda p: {prefix[k] + k: [v] for k, v in p.items()}
        _combine = lambda a, b: (lambda c: [c, c.update(b)][0])(dict(a))
        # Save images by iterating over the different views and surfaces
        for view, vparams in views.items():
            # Combine basic, view, and surface parameters
            params = _combine(basic, vparams)
            # Set the view
            handle._set_view(**_tolists(params))
            # Save image
            if "viol" in nm or "TRMI" in nm:
                filename = file_pattern.format(base=subject+f"_{surprise_analysis_type}", view=view, nm=nm)
            else:
                filename = file_pattern.format(base=subject, view=view, nm=nm)

            output_path = os.path.join(
                figure_result_dir, filename
            )
            handle.getImage(output_path, size=(3840, 2160))
            # the block below trims the edges of the image:
            # wait for image to be written
            while not os.path.exists(output_path):
                pass
            time.sleep(1.5)
            # try:
            #     import subprocess
            #     subprocess.call(['convert', '-trim', output_path, output_path])
            # except:
            #     pass
        # Close the window!
        handle.close()

/Users/dionysus/anaconda3/envs/prfexpectpub/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


creating web view
Generating new ctm file...
wm
wm
inflated
inflated
Started server on port 39076
Unknown parameter surface.curvature.contrast!
Stopping server
Generating new ctm file...
wm
wm
inflated
inflated
Started server on port 12481
Unknown parameter surface.curvature.contrast!


Stopping server
